# Interrupt + Human Approval — Pause for High-Risk Actions
## Stop Before High-Risk Actions — Interrupt and Resume Pattern
⏱ ~10 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/79-interrupt-human-approval/interrupt_human_approval_workbook.ipynb)

Uses LangGraph interrupt() to pause before executing risky actions. The graph stops at await_approval, serializes state to MemorySaver, and resumes when you call app.invoke() with Command(resume=True/False).

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why HITL matters for irreversible actions; interrupt vs breakpoint |
| 2 | **interrupt()** — Raises GraphInterrupt — graph suspends, state saved to checkpointer |
| 3 | **MemorySaver** — In-process checkpointer that holds state across invoke calls |
| 4 | **Resume** — Command(resume=True) approves; Command(resume=False) rejects |
| 5 | **Full Demo** — 5 actions (low/medium/high risk) with approval flow |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

## Part 1 — Human-in-the-Loop Concepts

### The problem with fully autonomous agents

Autonomous agents are efficient but fragile at decision boundaries. For high-stakes actions — deleting data, sending emails, spending money, modifying production systems — you want a human to review before execution.

LangGraph solves this with `interrupt()`: a node can pause mid-execution, yield a question to the caller, and wait for an explicit approval before continuing.

### The interrupt/resume lifecycle

```
invoke(initial_state) → graph runs → hits interrupt() → PAUSES, returns snapshot
                                                              ↓
                                               human reviews snapshot
                                                              ↓
invoke(Command(resume=decision)) → graph continues from the interrupt point → END
```

The graph state is serialized (checkpointed) between the two `invoke()` calls. This lets the pause last for seconds, minutes, or days.

### Checkpointer: the persistence layer

An `interrupt()` only works when the graph has a **checkpointer**. The checkpointer serializes state after each node and restores it on `Command(resume=...)`.

| Checkpointer | Persistence | Use case |
|-------------|-------------|----------|
| `MemorySaver` | In-process RAM | Development, notebooks |
| `SqliteSaver` | Local SQLite file | Single-machine production |
| `AsyncRedisSaver` | Redis | Multi-process, distributed |

> **Prerequisite:** Familiar with LangGraph `StateGraph` and `compile()` (examples 62–65).

In [ ]:
from typing import TypedDict
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

# simple approval demo — no LLM needed, just the interrupt pattern
class ApprovalState(TypedDict):
    action:   str
    risk:     str
    approved: bool
    result:   str

def propose_action(state: ApprovalState) -> dict:
    # just a named node for observability — logs the action before the interrupt
    print(f"  [propose_action] queuing: {state['action']}")
    return {}

def await_approval(state: ApprovalState) -> dict:
    # interrupt() serializes state here and returns the payload to the caller
    # graph resumes when Command(resume=<value>) is sent
    decision = interrupt({"action": state["action"], "risk": state["risk"],
                          "question": "Approve this action?"})
    if decision:
        return {"approved": True,  "result": f"EXECUTED: {state['action']}"}
    return  {"approved": False, "result": f"REJECTED: {state['action']}"}

g = StateGraph(ApprovalState)
g.add_node("propose_action", propose_action)
g.add_node("await_approval", await_approval)
g.add_edge(START,            "propose_action")
g.add_edge("propose_action", "await_approval")
g.add_edge("await_approval", END)

# MemorySaver: in-process checkpointer; no external dependency needed for notebooks
checkpointer = MemorySaver()
app = g.compile(checkpointer=checkpointer)

# --- demo: one action, manual approve ---
cfg = {"configurable": {"thread_id": "demo-1"}}
print("Step 1: invoke to reach the interrupt point")
app.invoke({"action": "Send welcome email", "risk": "low",
            "approved": False, "result": ""}, config=cfg)

print("\nStep 2: simulate human approval → resume")
final = app.invoke(Command(resume=True), config=cfg)
print(f"Result: {final['result']}")

## Part 2 — Risk-Based Auto-Routing

Not every action needs human review. You can implement a **risk policy** that:
- Auto-approves low-risk actions (no interrupt)
- Routes medium-risk actions to a human queue (interrupt)
- Rejects high-risk actions automatically without running them

This is done with a **conditional edge** that reads the `risk` field from state before deciding whether to route to the interrupt node or skip directly to execution.

### Graph structure

```
START → classify_risk → [low: execute_node] → END
                      → [medium/high: await_approval → execute_node] → END
```

`add_conditional_edges` takes a function that returns a node name string based on state.

In [ ]:
SAMPLE_ACTIONS = [
    {"action": "Send welcome email to new user",             "risk": "low"},
    {"action": "Archive Q1 reports",                         "risk": "low"},
    {"action": "Reset user password",                        "risk": "medium"},
    {"action": "Delete inactive accounts older than 1 year", "risk": "high"},
    {"action": "Drop all temp tables in production DB",      "risk": "high"},
]

class RiskState(TypedDict):
    action:   str
    risk:     str
    approved: bool
    result:   str

def route_by_risk(state: RiskState) -> str:
    # conditional edge function: return next node name based on risk level
    if state["risk"] == "low":
        return "execute_node"   # skip the interrupt entirely
    if state["risk"] == "medium":
        return "await_approval"
    return "reject_node"        # high risk is denied without execution

def execute_node(state: RiskState) -> dict:
    if state["approved"]:
        return {"result": f"EXECUTED AFTER APPROVAL: {state['action']}"}
    return {"approved": True, "result": f"AUTO-EXECUTED: {state['action']}"}

def reject_node(state: RiskState) -> dict:
    return {"approved": False, "result": f"AUTO-REJECTED: {state['action']}"}

def await_approval_risk(state: RiskState) -> dict:
    decision = interrupt({"action": state["action"], "risk": state["risk"]})
    if decision:
        return {"approved": True,  "result": f"APPROVED: {state['action']}"}
    return  {"approved": False, "result": f"REJECTED: {state['action']}"}

def route_after_approval(state: RiskState) -> str:
    return "execute_node" if state["approved"] else "reject_node"

g2 = StateGraph(RiskState)
g2.add_node("await_approval", await_approval_risk)
g2.add_node("execute_node",   execute_node)
g2.add_node("reject_node",    reject_node)
g2.add_conditional_edges(
    START,
    route_by_risk,
    {"execute_node": "execute_node", "await_approval": "await_approval", "reject_node": "reject_node"},
)
g2.add_conditional_edges("await_approval", route_after_approval)
g2.add_edge("execute_node", END)
g2.add_edge("reject_node", END)

ck2  = MemorySaver()
app2 = g2.compile(checkpointer=ck2)

# simulate: low=auto, medium=approve, high=auto-reject
POLICY = {"low": None, "medium": True, "high": None}

for i, action_def in enumerate(SAMPLE_ACTIONS):
    cfg  = {"configurable": {"thread_id": f"risk-{i}"}}
    init = {**action_def, "approved": False, "result": ""}

    result = app2.invoke(init, config=cfg)
    decision = POLICY[action_def["risk"]]

    if decision is None:
        # low risk: graph ran to END without hitting interrupt
        print(f"[{action_def['risk'].upper():6}] {action_def['action']}")
        print(f"  -> {result['result']}")
    else:
        # medium/high: hit the interrupt, now resume
        final = app2.invoke(Command(resume=decision), config=cfg)
        print(f"[{action_def['risk'].upper():6}] {action_def['action']}")
        print(f"  Human decision: {'APPROVE' if decision else 'REJECT'}")
        print(f"  -> {final['result']}")
    print()

## Part 3 — Approval with Edits

A richer HITL pattern lets the human not just approve/reject, but also *edit* the action before it runs. The `interrupt()` payload can return any value — including a modified action string.

The `Command(resume=value)` value becomes the return value of `interrupt()` inside the node. So if the human sends back a dict like `{"decision": True, "edited_action": "Archive Q1 reports (dry run)"}`, the node reads it and executes the modified version.

This mirrors how real HITL systems work:
- AI proposes an action
- Human reviews and optionally tweaks parameters
- Revised action is executed

In [ ]:
class EditableState(TypedDict):
    action:          str
    risk:            str
    final_action:    str   # may differ from action if human edits
    approved:        bool
    result:          str

def await_editable_approval(state: EditableState) -> dict:
    # human can return {"decision": True/False, "edited_action": "..."} or just True/False
    response = interrupt({
        "action":   state["action"],
        "risk":     state["risk"],
        "question": "Approve? You may return {'decision': True/False, 'edited_action': '...'} to modify.",
    })

    if isinstance(response, dict):
        decision      = response.get("decision", False)
        final_action  = response.get("edited_action", state["action"])
    else:
        decision     = bool(response)
        final_action = state["action"]

    if decision:
        return {"approved": True, "final_action": final_action,
                "result": f"EXECUTED (edited): {final_action}"}
    return {"approved": False, "final_action": state["action"],
            "result": f"REJECTED: {state['action']}"}

g3 = StateGraph(EditableState)
g3.add_node("await_approval", await_editable_approval)
g3.add_edge(START, "await_approval")
g3.add_edge("await_approval", END)
ck3  = MemorySaver()
app3 = g3.compile(checkpointer=ck3, interrupt_before=["await_approval"])

# scenario: human approves but edits the action
cfg = {"configurable": {"thread_id": "edit-demo-1"}}
app3.invoke({"action": "Delete inactive accounts older than 1 year",
             "risk": "high", "final_action": "", "approved": False, "result": ""}, config=cfg)

# human edits: change to 2 years to be safer
human_response = {"decision": True, "edited_action": "Delete inactive accounts older than 2 years"}
final = app3.invoke(Command(resume=human_response), config=cfg)
print("Human edited and approved:")
print(f"  Original: Delete inactive accounts older than 1 year")
print(f"  Final:    {final['final_action']}")
print(f"  Result:   {final['result']}")

## Part 4 — Audit Log and Timeout Pattern

In production, every approval decision needs an audit trail:
- Who approved (or rejected) the action
- When the decision was made
- What the final action was

The audit log lives in state alongside the action so it's checkpointed automatically and flows through the graph.

### Timeout simulation

In real systems, an interrupt can time out if no human responds within N minutes. In LangGraph you simulate this by checking a timestamp in state:

1. Record `proposed_at` timestamp when `propose_action` runs
2. In `await_approval`, check `time.time() - proposed_at > TIMEOUT_SECONDS`
3. If timed out, auto-reject (or escalate)

In [ ]:
import time
import datetime

TIMEOUT_SECONDS = 5   # low for demo — use 3600 (1 hour) in production

class AuditState(TypedDict):
    action:       str
    risk:         str
    proposed_at:  float
    approved:     bool
    approver:     str
    decided_at:   str
    result:       str

def propose_with_timestamp(state: AuditState) -> dict:
    return {"proposed_at": time.time()}

def await_with_audit(state: AuditState) -> dict:
    elapsed = time.time() - state["proposed_at"]
    if elapsed > TIMEOUT_SECONDS:
        return {"approved": False, "approver": "system",
                "decided_at": datetime.datetime.utcnow().isoformat(),
                "result": f"TIMED OUT after {elapsed:.0f}s: {state['action']}"}

    response = interrupt({"action": state["action"], "risk": state["risk"],
                          "pending_seconds": round(elapsed, 1)})

    if isinstance(response, dict):
        approved  = response.get("decision", False)
        approver  = response.get("approver", "unknown")
    else:
        approved  = bool(response)
        approver  = "auto"

    return {
        "approved":    approved,
        "approver":    approver,
        "decided_at":  datetime.datetime.utcnow().isoformat(),
        "result":      ("EXECUTED" if approved else "REJECTED") + f": {state['action']}",
    }

g4 = StateGraph(AuditState)
g4.add_node("propose", propose_with_timestamp)
g4.add_node("approve", await_with_audit)
g4.add_edge(START, "propose"); g4.add_edge("propose", "approve"); g4.add_edge("approve", END)
ck4  = MemorySaver()
app4 = g4.compile(checkpointer=ck4, interrupt_before=["approve"])

# normal approval path
cfg = {"configurable": {"thread_id": "audit-1"}}
app4.invoke({"action": "Reset user password", "risk": "medium",
             "proposed_at": 0.0, "approved": False, "approver": "", "decided_at": "", "result": ""}, config=cfg)

final = app4.invoke(Command(resume={"decision": True, "approver": "alice@example.com"}), config=cfg)
print("Audit record:")
for k in ["action", "approved", "approver", "decided_at", "result"]:
    print(f"  {k}: {final[k]}")

## Exercises

---

**Exercise 1 — Multi-Step Approval Chain**

Build a 2-step approval: `manager_approval` → `director_approval`. The director node only runs if the manager approved. Both use `interrupt()`. High-risk actions require both approvals; low-risk only need the manager.

---

**Exercise 2 — Approval with Comments**

Extend the editable approval pattern so the human can attach a comment:
`{"decision": True, "edited_action": "...", "comment": "Reviewed and approved — EVA"}`.

Store the comment in state and include it in the audit log.

---

**Exercise 3 — Timeout Test**

Set `TIMEOUT_SECONDS = 2`. Add a `time.sleep(3)` between the first `invoke()` and the `Command(resume=...)`. Verify the graph detects the timeout and auto-rejects without waiting for human input.

---

**Exercise 4 — Persistent Checkpointer**

Replace `MemorySaver()` with `SqliteSaver.from_conn_string(":memory:")` (in-memory SQLite). Verify the graph still resumes correctly. Then change to a file path like `"approvals.db"` — the approval queue now survives process restarts.

*Hint:* `from langgraph.checkpoint.sqlite import SqliteSaver`

---
## Answer Key

Attempt the exercises before reading below.

In [ ]:
# Answer 2 — Approval with Comments
class CommentedState(TypedDict):
    action:       str
    risk:         str
    final_action: str
    approved:     bool
    approver:     str
    comment:      str
    result:       str

def await_with_comment(state: CommentedState) -> dict:
    response = interrupt({
        "action": state["action"],
        "risk":   state["risk"],
        "prompt": "Return dict: {decision, approver, edited_action (opt), comment (opt)}",
    })

    if isinstance(response, dict):
        decision     = response.get("decision", False)
        final_action = response.get("edited_action", state["action"])
        approver     = response.get("approver", "unknown")
        comment      = response.get("comment", "")
    else:
        decision     = bool(response)
        final_action = state["action"]
        approver     = "auto"
        comment      = ""

    return {
        "approved":    decision,
        "final_action": final_action,
        "approver":    approver,
        "comment":     comment,
        "result":      ("EXECUTED" if decision else "REJECTED") + f": {final_action}",
    }

gc = StateGraph(CommentedState)
gc.add_node("approve", await_with_comment)
gc.add_edge(START, "approve"); gc.add_edge("approve", END)
ckc  = MemorySaver()
appc = gc.compile(checkpointer=ckc, interrupt_before=["approve"])

cfg = {"configurable": {"thread_id": "comment-1"}}
appc.invoke({"action": "Archive Q1 reports", "risk": "low",
             "final_action": "", "approved": False,
             "approver": "", "comment": "", "result": ""}, config=cfg)

final = appc.invoke(Command(resume={
    "decision": True,
    "approver": "alice@example.com",
    "comment":  "Verified with finance team — safe to proceed.",
}), config=cfg)

print("Audit record with comment:")
for k in ["action", "final_action", "approved", "approver", "comment", "result"]:
    print(f"  {k}: {final[k]}")

In [ ]:
# Answer 3 — Timeout Test
import time

TIMEOUT_TEST_SECONDS = 2

def await_with_timeout_test(state: AuditState) -> dict:
    elapsed = time.time() - state["proposed_at"]
    if elapsed > TIMEOUT_TEST_SECONDS:
        return {"approved": False, "approver": "system-timeout",
                "decided_at": datetime.datetime.utcnow().isoformat(),
                "result": f"TIMED OUT ({elapsed:.1f}s > {TIMEOUT_TEST_SECONDS}s): {state['action']}"}
    response = interrupt({"action": state["action"]})
    return {"approved": bool(response), "approver": "human",
            "decided_at": datetime.datetime.utcnow().isoformat(),
            "result": ("EXECUTED" if bool(response) else "REJECTED") + f": {state['action']}"}

gt = StateGraph(AuditState)
gt.add_node("propose", propose_with_timestamp)
gt.add_node("approve", await_with_timeout_test)
gt.add_edge(START, "propose"); gt.add_edge("propose", "approve"); gt.add_edge("approve", END)
ckt  = MemorySaver()
appt = gt.compile(checkpointer=ckt, interrupt_before=["approve"])

cfg = {"configurable": {"thread_id": "timeout-test-1"}}
appt.invoke({"action": "Drop temp tables", "risk": "high",
             "proposed_at": 0.0, "approved": False,
             "approver": "", "decided_at": "", "result": ""}, config=cfg)

print("Sleeping 3 seconds to simulate delayed human response...")
time.sleep(3)

final = appt.invoke(Command(resume=True), config=cfg)
print(f"Result: {final['result']}")
print(f"Approver: {final['approver']}")

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*